# Infer-8b-TrueSkill-Formules-Fermees : la mise a jour O(1) de Herbrich-Minka-Graepel

**Serie** : Programmation Probabiliste avec Infer.NET (8b/19)  
**Duree estimee** : 20 minutes  
**Prerequis** : [Infer-8-TrueSkill](Infer-8-TrueSkill.ipynb)

***

## Objectifs

- Deriver les fonctions de troncature V(t) et W(t) (cas a 2 joueurs)
- Implementer la mise a jour closed-form de TrueSkill -- O(1) par match
- Verifier numeriquement la coherence exacte avec le moteur EP d'Infer.NET
- Comprendre le terme de dynamique tau^2 (equilibre contraction / regrowth)

***

## Navigation

| précédent | Suivant |
|-----------|--------|
| [Infer-8-TrueSkill](Infer-8-TrueSkill.ipynb) | [Infer-9-Classification](Infer-9-Classification.ipynb) |

***

## Sources et références canoniques

- **TrueSkill** — Herbrich, R., Minka, T. & Graepel, T. (2007). *TrueSkill(TM): A Bayesian Skill Rating System*. NeurIPS 2007, Microsoft Research Cambridge. La matière première de cette lettre : les formes fermées V(t), W(t) de la mise à jour à 2 joueurs.


## 1. La vraie contribution algorithmique du papier TrueSkill

Tout au long d'[Infer-8-TrueSkill](Infer-8-TrueSkill.ipynb), nous avons laissé le **moteur d'Expectation Propagation (EP)** d'Infer.NET calculer les postérieurs des skills. C'est rigoureux, mais cela masque **la vraie contribution algorithmique du papier TrueSkill** : dans le cas à 2 joueurs, la mise à jour admet une **forme fermée exacte** — O(1) par match, sans aucune inférence itérative. C'est ce qui rend TrueSkill déployable en production sur des millions de joueurs (Xbox Live), où relancer EP à chaque match serait prohibitif.

> **Source primaire** : Herbrich, Minka & Graepel (2007), *TrueSkill(TM): A Bayesian Skill Rating System* (NeurIPS / Microsoft Research Cambridge). Le jumeau [PyMC-08-TrueSkill](../PyMC/PyMC-08-TrueSkill.ipynb) expose la même dérivation (section 7 bis).

### Le problème : une vraisemblance « inégale » non-Gaussienne

La contrainte $\text{perf}_w > \text{perf}_l$ est une **inégalité** : le postérieur qu'elle induit est une Gaussienne **tronquée**, donc non-Gaussien. Or on veut maintenir chaque skill comme une Gaussienne $\mathcal{N}(\mu, \sigma^2)$ (pour le réinjecter comme prior au match suivant). EP résout cela en **projetant** le postérieur tronqué sur la meilleure Gaussienne approximante (moment matching sur le graphe de facteurs).

### La mise à jour closed-form à 2 joueurs (fonctions de troncature V, W)

Après convergence d'EP sur le facteur « gagnant/perdant », la mise à jour se ramène à deux fonctions auxiliaires — troncatures de la Gaussienne centrée réduite :

$$c = \sqrt{2\beta^2 + \sigma_w^2 + \sigma_l^2}, \qquad t = \frac{\mu_w - \mu_l}{c}$$

$$V(t) = \frac{\mathcal{N}(t;\,0,1)}{\Phi(t)}, \qquad W(t) = V(t)\big(V(t) + t\big)$$

où $\mathcal{N}$ est la densité et $\Phi$ la CDF de la Gaussienne centrée réduite. Les mises à jour du gagnant ($w$) et du perdant ($l$) deviennent alors :

$$\mu_w' = \mu_w + \frac{\sigma_w^2}{c}\,V(t), \qquad \mu_l' = \mu_l - \frac{\sigma_l^2}{c}\,V(t)$$

$$(\sigma_w^2)' = \sigma_w^2\!\left(1 - \frac{\sigma_w^2}{c^2}\,W(t)\right), \qquad (\sigma_l^2)' = \sigma_l^2\!\left(1 - \frac{\sigma_l^2}{c^2}\,W(t)\right)$$

> **Détail subtil** : la variance se contracte du facteur $\frac{\sigma^2}{c^2}\,W(t)$, et **non** de $W(t)$ seul. Le ratio $\sigma^2/c^2$ borne la contraction : un prior très confiant ($\sigma^2 \ll c^2$) ne peut guère se resserrer davantage.

**Interprétation** : $V(t)$ mesure à quel point le match a été informatif. Un match équilibré ($t \approx 0$) instruit beaucoup ($V(0) \approx 0{,}80$) ; une victoire attendue ($t \gg 0$) instruit peu. La variance se contracte en proportion.

### La dynamique entre les matchs : le terme $\tau^2$

En production, TrueSkill **ajoute** du bruit au skill avant chaque match : $\sigma^2 \leftarrow \sigma^2 + \tau^2$. L'incertitude ne décroît donc pas indéfiniment — elle atteint un équilibre entre la contraction (information du match) et la régrowth ($\tau^2$). Un joueur inactif voit son incertitude **augmenter**, ce qui justifie de repondérer rapidement ses premiers matchs de retour (extension dynamique d'Infer-8, section 5).


In [1]:
#r "nuget: MathNet.Numerics"


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages MathNet.Numerics, 5.0.0

In [2]:
// Demonstration numerique : la mise a jour closed-form EP (V(t), W(t)) que le moteur
// Infer.NET d'Infer-8 (sections 3-9) calcule sous le capot. Ecrite ici "a la main" (forme fermee
// de Herbrich-Minka-Graepel, NeurIPS 2007) pour verifier la coherence avec Infer.NET.
using MathNet.Numerics.Distributions;
using System;

// Memes parametres qu'a l'initialisation (Infer-8, section 3) : mu=25, sigma=25/3, beta=sigma/2
double muW = 25.0, muL = 25.0;            // match entre deux joueurs de skill identique (priors egaux)
double sigW = 25.0/3.0, sigL = 25.0/3.0;  // sigma initial
double beta = (25.0/3.0)/2.0;             // variabilite de la performance

// Fonctions de troncature de la Gaussienne centree reduite (Herbrich-Minka-Graepel 2007)
var stdNormal = new Normal(0.0, 1.0);
double c = Math.Sqrt(2*beta*beta + sigW*sigW + sigL*sigL);     // denominateur commun c
double t = (muW - muL)/c;                                       // ecart standardise
double Vt = stdNormal.Density(t) / stdNormal.CumulativeDistribution(t); // V(t) = phi(t)/Phi(t)
double Wt = Vt*(Vt + t);                                        // W(t) = V(t)(V(t)+t)

// Mises a jour closed-form (formes fermees de Herbrich-Minka-Graepel 2007).
// Note : la variance se contracte du facteur (sigma^2 / c^2) * W(t), pas de W(t) seul --
// ce facteur d'echelle est indispensable pour retrouver le posterior d'Infer.NET (Infer-8, section 3).
double muWNew = muW + sigW*sigW*Vt/c;
double muLNew = muL - sigL*sigL*Vt/c;
double sigWNew = Math.Sqrt(sigW*sigW*(1.0 - (sigW*sigW/(c*c))*Wt));
double sigLNew = Math.Sqrt(sigL*sigL*(1.0 - (sigL*sigL/(c*c))*Wt));

Console.WriteLine("Mise a jour closed-form EP sur un match (priors egaux, mu=25, sigma=8,33)");
Console.WriteLine($"  c = {c:F4}   t = {t:F4}   V(t) = {Vt:F4}   W(t) = {Wt:F4}");
Console.WriteLine($"  GAGNANT : mu {muW:F2} -> {muWNew:F2}   sigma {sigW:F2} -> {sigWNew:F2}");
Console.WriteLine($"  PERDANT  : mu {muL:F2} -> {muLNew:F2}   sigma {sigL:F2} -> {sigLNew:F2}");
Console.WriteLine("Verification de coherence : ces valeurs (mu 29,21 / 20,79 ; sigma 7,19)");
Console.WriteLine("reproduisent exactement le posterior Infer.NET de la section 3 d'Infer-8 (cellule 16) --");
Console.WriteLine("le moteur EP et la forme fermee de Herbrich 2007 coincident sur ce cas 2 joueurs.");


Mise a jour closed-form EP sur un match (priors egaux, mu=25, sigma=8,33)


  c = 13,1762   t = 0,0000   V(t) = 0,7979   W(t) = 0,6366


  GAGNANT : mu 25,00 -> 29,21   sigma 8,33 -> 7,19


  PERDANT  : mu 25,00 -> 20,79   sigma 8,33 -> 7,19


Verification de coherence : ces valeurs (mu 29,21 / 20,79 ; sigma 7,19)


reproduisent exactement le posterior Infer.NET de la section 3 d'Infer-8 (cellule 16) --


le moteur EP et la forme fermee de Herbrich 2007 coincident sur ce cas 2 joueurs.


## 2. Lecture : EP par Infer.NET vs forme fermée — deux routes, le même postérieur

La cellule précédente calcule en forme fermée (O(1), sans compilation ni échantillonnage) ce qu'Infer.NET obtient en lançant son moteur EP en section 3 d'[Infer-8-TrueSkill](Infer-8-TrueSkill.ipynb) (cellule 16). Sur un match à priors égaux, l'écart standardisé est nul ($t = 0$) : le match est maximalement informatif ($V(0) \approx 0{,}80$), d'où un déplacement du skill de $\pm 4{,}21$ points et une contraction de l'incertitude de $8{,}33$ à $7{,}19$ ($-14\,\%$). **On retrouve exactement le postérieur Infer.NET** `N(29,21, 7,19)` / `N(20,79, 7,19)` de la cellule 16 d'Infer-8 — la forme fermée et le moteur EP coïncident sur ce cas canonique à 2 joueurs.

C'est cette exactitude O(1) qui rend TrueSkill déployable à l'échelle de millions de joueurs : chaque mise à jour coûte une poignée de multiplications plutôt qu'une inférence compilée. Le moteur EP d'Infer.NET redevient indispensable dès que le modèle s'écarte du cas canonique — matchs nuls via `ConstrainBetween` (section 4 d'Infer-8), équipes (section 6), multi-joueurs (section 7) — cas pour lesquels il n'existe pas toujours de forme fermée. Les deux approches sont **complémentaires** : EP pour la flexibilité du modèle, forme fermée pour la vélocité en production.


## Synthèse : deux routes, le même postérieur

La forme fermée de Herbrich-Minka-Graepel (2007) isole la contribution algorithmique du cas à 2 joueurs : deux fonctions de troncature V(t) et W(t), une poignée d'opérations arithmétiques, et la mise à jour complète d'un match — sans compilation ni échantillonnage. La démonstration numérique retrouve exactement le postérieur du moteur EP (`N(29,21, 7,19)` sur priors égaux) : O(1) par match, la vitesse de production d'Xbox Live.

Le moteur EP redevient l'outil général dès que le modèle s'écarte du cas canonique (matchs nuls, équipes, free-for-all). Les deux approches sont complémentaires : EP pour la flexibilité du modèle, forme fermée pour la vélocité en production.
